In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `03_gold`.kpi;
CREATE OR REPLACE TABLE `03_gold`.kpi.curated_sales AS
SELECT
    f.order_number,
    f.line_item,
    f.order_date,
    f.year,
    f.month,
    f.customerkey,
    COALESCE(c.gender, 'Unkown') AS gender,
    COALESCE(c.continent,'Unkown') AS continent,
    f.storekey,
    COALESCE(s.store_country, 'Online') AS country,
    f.productkey,
    p.category,
    p.subcategory,
    f.quantity,
    f.unit_price_usd,
    f.exchange,
    f.revenue_usd,
    f.delivery_days,
    f.channel

FROM `03_gold`.fact_tables.fact_sales f
LEFT JOIN `03_gold`.dim_tables.dim_customer c
    ON f.customerkey = c.customerkey
LEFT JOIN `03_gold`.dim_tables.dim_products p
    ON f.productkey = p.productkey
LEFT JOIN `03_gold`.dim_tables.dim_stores s
    ON f.storekey = s.storekey;

In [0]:
%sql
SELECT * FROM `03_gold`.kpi.curated_sales;

In [0]:
%sql
with revenue_of_year as(
  SELECT
      year,
      month,
      ROUND(SUM(revenue_usd),2) AS revenue_usd
  FROM `03_gold`.kpi.curated_sales
  where year = '2021'
  GROUP BY year, month
  ORDER BY year, month
)
select * from revenue_of_year

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH monthly AS (
    SELECT month,year, SUM(revenue_usd) AS revenue
    FROM `03_gold`.kpi.curated_sales
    where year = 2021
    GROUP BY month,year
),
total AS (
    SELECT SUM(revenue) AS total_rev FROM monthly
)
SELECT
    month,
    year,
    ROUND(revenue,2) as monthly_revenue,
    ROUND(revenue * 100 / total_rev,2) AS percent
FROM monthly, total
ORDER BY revenue DESC
LIMIT 3;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH peak_months AS (
    SELECT month
    FROM (
        SELECT month, SUM(revenue_usd) AS rev
        FROM `03_gold`.kpi.curated_sales
        WHERE year = 2021
        GROUP BY month
        ORDER BY rev DESC
        LIMIT 3
    )
)
SELECT
    category,
    ROUND(SUM(revenue_usd),2) AS peak_month_revenue,
    ROUND(
        SUM(revenue_usd)*100 / SUM(SUM(revenue_usd)) OVER(),2
    ) AS percent_of_peak_total
FROM `03_gold`.kpi.curated_sales
WHERE year = 2021
  AND month IN (SELECT month FROM peak_months)
GROUP BY category
ORDER BY peak_month_revenue DESC
LIMIT 3;

In [0]:
%sql

SELECT
    ROUND(AVG(delivery_days),2) AS avg_days,
    COUNT(*) AS total_orders
FROM `03_gold`.kpi.curated_sales
WHERE delivery_days IS NOT NULL;

In [0]:
%sql

SELECT
    country,
    ROUND(AVG(delivery_days),2) AS avg_days,
    COUNT(*) AS order_count,
    PERCENTILE(delivery_days,0.5) AS median_days
FROM `03_gold`.kpi.curated_sales
WHERE delivery_days IS NOT NULL
GROUP BY country
ORDER BY avg_days DESC
LIMIT 5;
     

In [0]:
%sql

SELECT
    continent,

    ROUND(
        SUM(CASE WHEN channel='online' THEN revenue_usd END) /
        NULLIF(COUNT(DISTINCT CASE WHEN channel='online' THEN order_number END),0)
    ,2) AS aov_online,

    ROUND(
        SUM(CASE WHEN channel='store' THEN revenue_usd END) /
        NULLIF(COUNT(DISTINCT CASE WHEN channel='store' THEN order_number END),0)
    ,2) AS aov_store,

    COUNT(DISTINCT CASE WHEN channel='online' THEN order_number END) AS online_orders,
    COUNT(DISTINCT CASE WHEN channel='store' THEN order_number END) AS store_orders

FROM `03_gold`.kpi.curated_sales
WHERE continent <> "Unkown"
GROUP BY continent;
     

In [0]:
%sql
SELECT
    ROW_NUMBER() OVER(ORDER BY SUM(quantity) DESC) AS rank,
    category,
    SUM(quantity) AS units_sold,
    ROUND(SUM(quantity)*100 / SUM(SUM(quantity)) OVER(),2) AS percent
FROM `03_gold`.kpi.curated_sales
GROUP BY category
LIMIT 5;

In [0]:
%sql

SELECT
    ROW_NUMBER() OVER(ORDER BY SUM(revenue_usd) DESC) AS rank,
    category,
    ROUND(SUM(revenue_usd),2) AS revenue,
    ROUND(SUM(revenue_usd)*100 / SUM(SUM(revenue_usd)) OVER(),2) AS percent
FROM `03_gold`.kpi.curated_sales
GROUP BY category
LIMIT 5;

In [0]:
%sql
SELECT
    continent,
    gender,
    COUNT(DISTINCT customerkey) AS customer_count,
    ROUND(SUM(quantity * unit_price_usd * exchange), 2) AS total_spent
from `03_gold`.kpi.curated_sales
where continent <> "Unkown"
GROUP BY continent, gender;

In [0]:
%sql
WITH cust_orders AS (
    SELECT
        continent,
        customerkey,
        COUNT(DISTINCT order_number) AS orders
    FROM `03_gold`.kpi.curated_sales
    GROUP BY continent, customerkey
)
SELECT
    continent,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN orders >=2 THEN 1 ELSE 0 END) AS repeat_customers,
    ROUND(
        SUM(CASE WHEN orders >=2 THEN 1 ELSE 0 END)*100/COUNT(*),2
    ) AS repeat_rate
FROM cust_orders
WHERE continent <> "Unkown"
GROUP BY continent;
     